# Customer Cohort Retention and RFM Segmentation
## Exploratory Data Analysis and Feature Engineering

### Project Objective
This notebook covers the processing steps used to analyze retail transactions. The goal is to track monthly customer retention and group buyers into marketing segments based on Recency, Frequency, and Monetary value.

## 1. Environment and Data Loading

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate raw transactions CSV file
raw_path = os.path.join("..", "data", "raw", "online_retail.csv")

df_raw = pd.read_csv(raw_path)
print(f"Raw dataset dimensions: {df_raw.shape[0]:,} rows by {df_raw.shape[1]} columns")
df_raw.head(3)

## 2. Missing Value Profiling and Cancellation Handling
The dataset contains guest transactions without a customer identifier. It also records returns and cancellations as negative order quantities or invoice numbers starting with C.

In [ ]:
# Parse invoice timestamps
df = df_raw.copy()
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Flag cancellations and return orders
df['IsCancelled'] = df['InvoiceNo'].astype(str).str.startswith('C') | (df['Quantity'] < 0)
print(f"Cancelled transaction records: {df['IsCancelled'].sum():,}")

# Assign Guest label to missing customer identifiers
missing_cust = df['CustomerID'].isna().sum()
print(f"Missing customer identifier rows: {missing_cust:,}")

df['CustomerID'] = df['CustomerID'].fillna(-1).astype(int).astype(str)
df['CustomerID'] = df['CustomerID'].replace('-1', 'Guest')
df['Description'] = df['Description'].fillna('Unknown Product').str.strip()

## 3. Financial Metrics and Outlier Capping
We calculate total sales revenue and apply 99.9th percentile capping to remove data entry errors in prices and order sizes.

In [ ]:
# Determine 99.9th percentile thresholds on positive transactions
q_limit = df[df['Quantity'] > 0]['Quantity'].quantile(0.999)
p_limit = df[df['UnitPrice'] > 0]['UnitPrice'].quantile(0.999)
print(f"Quantity threshold (99.9th percentile): {q_limit:.0f} units")
print(f"Unit price threshold (99.9th percentile): ${p_limit:.2f}")

# Clip extreme outliers
df['Quantity'] = np.clip(df['Quantity'], -q_limit, q_limit)
df['UnitPrice'] = np.clip(df['UnitPrice'], 0, p_limit)

# Calculate revenue metrics
df['TotalSales'] = df['Quantity'] * df['UnitPrice']
df['COGS'] = df['Quantity'] * (df['UnitPrice'] * 0.60)
df['Profit'] = df['TotalSales'] - df['COGS']

## 4. Cohort Retention Matrix
We group customers by their first purchase month (acquisition cohort) and measure how many return to buy in subsequent months.

In [ ]:
# Exclude guest checkouts and cancelled orders
df_active = df[(df['CustomerID'] != 'Guest') & (~df['IsCancelled'])].copy()

# Determine order month and cohort acquisition month
df_active['OrderMonth'] = df_active['InvoiceDate'].dt.to_period('M')
df_active['CohortMonth'] = df_active.groupby('CustomerID')['OrderMonth'].transform('min')

# Calculate month offset distance from initial purchase
df_active['CohortIndex'] = (
    (df_active['OrderMonth'].dt.year - df_active['CohortMonth'].dt.year) * 12 +
    (df_active['OrderMonth'].dt.month - df_active['CohortMonth'].dt.month)
)

# Build cohort counts table
cohort_group = df_active.groupby(['CohortMonth', 'CohortIndex'])['CustomerID'].nunique().reset_index()
cohort_pivot = cohort_group.pivot(index='CohortMonth', columns='CohortIndex', values='CustomerID')
cohort_pivot.index = cohort_pivot.index.astype(str)

# Compute percentage retention
cohort_sizes = cohort_pivot.iloc[:, 0]
retention_matrix = cohort_pivot.divide(cohort_sizes, axis=0) * 100

print("Sample Cohort Retention Matrix (%):")
retention_matrix.iloc[:6, :6].round(1)

## 5. Recency, Frequency, and Monetary (RFM) Segmentation
We aggregate purchase history per customer and assign quintile scores (1 to 5) across three metrics:
- Recency: Days since last completed purchase (lower days get score 5)
- Frequency: Total distinct order invoices (higher count gets score 5)
- Monetary: Total lifetime revenue spend (higher spend gets score 5)

In [ ]:
reference_date = df_active['InvoiceDate'].max()

# Aggregate metrics per customer
df_rfm = df_active.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalSales', 'sum')
).reset_index()

# Assign quintile scores
df_rfm['R_Score'] = pd.qcut(df_rfm['Recency'], q=5, labels=[5, 4, 3, 2, 1], duplicates='drop').astype(int)
df_rfm['F_Score'] = pd.qcut(df_rfm['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
df_rfm['M_Score'] = pd.qcut(df_rfm['Monetary'], q=5, labels=[1, 2, 3, 4, 5]).astype(int)
df_rfm['RFM_Score'] = df_rfm['R_Score'].astype(str) + df_rfm['F_Score'].astype(str) + df_rfm['M_Score'].astype(str)

# Classify customers into marketing segments
def classify_segment(row):
    r = row['R_Score']
    f = row['F_Score']
    m = row['M_Score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3 and m >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'New/Promising'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    else:
        return 'Hibernating/Lost'

df_rfm['Segment'] = df_rfm.apply(classify_segment, axis=1)

print("Customer Count by Segment:")
print(df_rfm['Segment'].value_counts())

## 6. Retention Heatmap and Segment Distribution Visualizations

In [ ]:
# Plot Segment Counts
fig, ax = plt.subplots(figsize=(8, 4))
counts = df_rfm['Segment'].value_counts()
bars = ax.barh(counts.index, counts.values, color='#1856FF')
ax.set_title("Customer Distribution by RFM Segment", fontsize=13, pad=10, fontweight='bold')
ax.set_xlabel("Number of Customers", fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for bar in bars:
    val = bar.get_width()
    ax.text(val + 30, bar.get_y() + bar.get_height()/2.0, f"{int(val):,}", va='center')

plt.tight_layout()
plt.show()

### Strategic Takeaways
1. The business relies heavily on the Champions segment for revenue stability. Protecting this group from churn has high financial priority.
2. Customers in the At Risk category have past purchase frequency but have not placed an order recently. Automated win-back campaigns should focus on this group.
3. First-month cohort retention drops sharply across all cohorts, indicating an opportunity to improve onboarding and post-purchase follow-up.